In [1]:
import os
from dotenv import load_dotenv

use_databricks = False

if use_databricks:
    os.environ["OPENAI_API_KEY"] = dbutils.secrets.get(
        scope="MainSecretScope", key="OPENAI_API_KEY"
    )
else:
    load_dotenv()

In [13]:
from langchain.agents import create_agent

provider = "google_genai"  # "google_genai" / "openai"

if provider == "google_genai":
    model = "google_genai:gemini-2.5-flash"
else:
    model = "gpt-5-mini"

agent = create_agent(model)

In [11]:
def moderate_ad(ad_text):

    prompt = f"""
        You are a professional content moderation system.

        Analyze the following advertisement text:

        "{ad_text}"

        Return ONLY valid JSON in this format:

        {{
        "is_gibberish": true/false,
        "is_spam": true/false,
        "is_inappropriate": true/false,
        "language": "detected language",
        "confidence": 0.0-1.0,
        "reason": "short explanation"
        }}
        """

    response = agent.invoke({
        "messages": [
            {"role": "user", "content": prompt}
        ]
    })

    return response["messages"][-1].content

In [15]:
ads = [
    "Brand new iPhone 15, sealed, warranty included.",
    # "Fr33 $$$ CLICK NOW LIMITED OFFER!!!!",
    # "asdkjhasdkjh123123",
    # "You are stupid and this is trash."
]

for ad in ads:
    result = moderate_ad(ad)
    print("Ad:", ad)
    print("Result:", result)
    print("----")

Ad: Brand new iPhone 15, sealed, warranty included.
Result: ```json
{
"is_gibberish": false,
"is_spam": true,
"is_inappropriate": false,
"language": "English",
"confidence": 0.98,
"reason": "The text is a clear advertisement for a product (iPhone 15), which is categorized as spam in content moderation. It is not gibberish or inappropriate."
}
```
----
